In [1]:
import os
import re
import glob
from collections import defaultdict, Counter

import numpy as np
import pandas as pd


# ----------------------------
# Tree printer
# ----------------------------
def print_tree_dir(path, prefix=""):
    items = sorted(os.listdir(path))
    for i, item in enumerate(items):
        full = os.path.join(path, item)
        connector = "├── " if i < len(items) - 1 else "└── "
        print(prefix + connector + item)
        if os.path.isdir(full):
            new_prefix = prefix + ("│   " if i < len(items) - 1 else "    ")
            print_tree_dir(full, new_prefix)


# ----------------------------
# Discovery helpers
# ----------------------------
def find_experiments(root):
    """Experiment dirs containing progress.csv (walk recursively)."""
    exps = []
    for r, _, files in os.walk(root):
        if "progress.csv" in files:
            exps.append(r)
    return sorted(exps)


def extract_base_name(name):
    """'XYZ - 3' -> 'XYZ' (group name)."""
    return name.split(" - ")[0] if " - " in name else name


def guess_algo(group):
    """Best-effort algo: prefix before first '-'."""
    return group.split("-", 1)[0] if "-" in group else group


# ----------------------------
# Reward + AUC from progress.csv
# ----------------------------
def compute_reward_and_auc(root, reward_col="episode_reward_mean"):
    reward_rows, auc_rows = [], []

    for exp in find_experiments(root):
        seed = os.path.basename(exp)
        group = extract_base_name(seed)
        algo = guess_algo(group)

        csv = os.path.join(exp, "progress.csv")
        try:
            df = pd.read_csv(csv)
        except Exception:
            continue

        if reward_col not in df.columns:
            continue

        rewards = pd.to_numeric(df[reward_col], errors="coerce").dropna()
        if rewards.empty:
            continue

        reward_rows.append({
            "algo": algo,
            "group": group,
            "seed": seed,
            "mean_reward": float(rewards.mean())
        })
        auc_rows.append({
            "algo": algo,
            "group": group,
            "seed": seed,
            "auc": float(np.trapz(rewards.values))
        })

    reward_df = pd.DataFrame(reward_rows)
    auc_df = pd.DataFrame(auc_rows)

    if reward_df.empty:
        reward_grp = pd.DataFrame(columns=["algo", "group", "mean", "std", "count", "mean_std"])
    else:
        reward_grp = reward_df.groupby(["algo", "group"])["mean_reward"].agg(["mean", "std", "count"]).reset_index()
        reward_grp["mean_std"] = reward_grp["mean"].round(3).astype(str) + " ± " + reward_grp["std"].round(3).astype(str)

    if auc_df.empty:
        auc_grp = pd.DataFrame(columns=["algo", "group", "mean", "std", "count", "mean_std"])
    else:
        auc_grp = auc_df.groupby(["algo", "group"])["auc"].agg(["mean", "std", "count"]).reset_index()
        auc_grp["mean_std"] = auc_grp["mean"].round(3).astype(str) + " ± " + auc_grp["std"].round(3).astype(str)

    return reward_grp, auc_grp, reward_df, auc_df


# ----------------------------
# TensorBoard tag utilities
# ----------------------------
RAY_PREFIX = re.compile(r"^(ray/(tune|train|rllib)/)")
def strip_ray(tag):
    return RAY_PREFIX.sub("", tag)

REWARD_PREF = [
    # "evaluation/episode_reward_mean",
    "episode_reward_mean",
    # "rollout/episode_reward_mean",
]
REWARD_REGEX = re.compile(r"(episode|ep).*(reward|return).*(mean|avg)", re.IGNORECASE)

X_PREF = [
    "timesteps_total",
    "num_env_steps_sampled",
    "num_agent_steps_sampled",
    "training_iteration",
]

def choose_reward_tag(tags):
    for t in REWARD_PREF:
        if t in tags:
            return t
    for t in tags:
        if REWARD_REGEX.search(t):
            return t
    return None

def choose_x_tag(tags):
    for t in X_PREF:
        if t in tags:
            return t
    return None

def get_event_files(d):
    return sorted(glob.glob(os.path.join(d, "**", "events.out.tfevents.*"), recursive=True))


# ----------------------------
# Smoothing + plotting
# ----------------------------
def _smooth(arr, sigma):
    if sigma is None or sigma <= 0:
        return arr
    try:
        from scipy.ndimage import gaussian_filter1d
        return gaussian_filter1d(arr, sigma=sigma)
    except Exception:
        # moving average fallback
        w = int(max(3, round(2 * sigma + 1)))
        if w % 2 == 0:
            w += 1
        kernel = np.ones(w) / w
        return np.convolve(arr, kernel, mode="same")


def _safe_minmax(values):
    """
    GUARANTEE:
      - vmin = min(values)
      - vmax = max(values)
      - if degenerate, widen slightly
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0

    vmin = float(np.min(values))
    vmax = float(np.max(values))

    if not np.isfinite(vmin) or not np.isfinite(vmax) or (vmax - vmin) < 1e-12:
        vmin -= 1.0
        vmax += 1.0

    return vmin, vmax


def _norm01(y, vmin, vmax):
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
    return np.clip((y - vmin) / denom, 0.0, 1.0)


def plot_raw(xs, m, s, title, save, smooth_sigma):
    import matplotlib.pyplot as plt
    m2, s2 = _smooth(m, smooth_sigma), _smooth(s, smooth_sigma)

    plt.figure(figsize=(12, 6))
    plt.plot(xs, m2)
    plt.fill_between(xs, m2 - s2, m2 + s2, alpha=0.2)
    plt.title(title)
    plt.grid(True)
    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.close()


def plot_norm(xs, m, s, title, save, smooth_sigma, vmin, vmax):
    import matplotlib.pyplot as plt
    m2, s2 = _smooth(m, smooth_sigma), _smooth(s, smooth_sigma)

    nm = _norm01(m2, vmin, vmax)
    denom = (vmax - vmin) if (vmax - vmin) != 0 else 1.0
    ns = s2 / denom
    lower = np.clip(nm - ns, 0, 1)
    upper = np.clip(nm + ns, 0, 1)

    plt.figure(figsize=(12, 6))
    plt.plot(xs, nm)
    plt.fill_between(xs, lower, upper, alpha=0.2)
    plt.ylim(0, 1)
    plt.title(title + f" (min-max: vmin={vmin:.6f}, vmax={vmax:.6f})")
    plt.grid(True)
    os.makedirs(os.path.dirname(save), exist_ok=True)
    plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.close()


# ----------------------------
# Charts: adaptável em X e Y
# ----------------------------
def make_charts(
    root,
    results_dir,
    smooth_sigma=3.0,
    norm_scope="global",  # "global" or "per_group"
):
    """
    Guarantees (as requested):
      - Normalized plot uses MIN-MAX
      - 0 == lowest value, 1 == highest value
      - scope controls whether min/max is computed globally or per group
    X axis:
      - uses x_tag if found (timesteps_total, training_iteration, ...)
      - otherwise uses event.step
    """
    try:
        from tensorflow.python.summary.summary_iterator import summary_iterator
    except Exception as e:
        print(f"[WARN] TensorFlow not available — skipping charts ({e})")
        return

    groups = defaultdict(list)
    for exp in find_experiments(root):
        groups[extract_base_name(os.path.basename(exp))].append(exp)

    if not groups:
        print("[WARN] No experiment folders found (no progress.csv).")
        return

    def extract_group_series(seed_dirs):
        event_files = []
        for sd in seed_dirs:
            event_files.extend(get_event_files(sd))
        if not event_files:
            return None

        # Detect tags
        tag_counts = Counter()
        for ef in event_files:
            try:
                for e in summary_iterator(ef):
                    if not hasattr(e, "summary") or e.summary is None:
                        continue
                    for v in e.summary.value:
                        tag_counts[strip_ray(v.tag)] += 1
            except Exception:
                pass

        y_tag = choose_reward_tag(tag_counts.keys())
        if not y_tag:
            return None

        x_tag = choose_x_tag(tag_counts.keys())  # may be None

        y_by_step = defaultdict(list)
        x_by_step = defaultdict(list) if x_tag else None

        for ef in event_files:
            try:
                for e in summary_iterator(ef):
                    if not hasattr(e, "summary") or e.summary is None:
                        continue
                    step = int(e.step)
                    for v in e.summary.value:
                        tag = strip_ray(v.tag)
                        if x_tag and tag == x_tag:
                            x_by_step[step].append(float(v.simple_value))
                        if tag == y_tag:
                            y_by_step[step].append(float(v.simple_value))
            except Exception:
                pass

        if not y_by_step:
            return None

        steps = sorted(y_by_step.keys())

        # xs: prefer x_tag if available, else step
        if x_tag:
            xs = np.array(
                [np.mean(x_by_step[s]) if s in x_by_step and len(x_by_step[s]) > 0 else float(s) for s in steps],
                dtype=float
            )
        else:
            xs = np.array([float(s) for s in steps], dtype=float)

        m = np.array([np.mean(y_by_step[s]) for s in steps], dtype=float)
        s = np.array([np.std(y_by_step[s]) for s in steps], dtype=float)

        order = np.argsort(xs)
        return xs[order], m[order], s[order], y_tag, x_tag

    # Cache everything first (needed for global min-max)
    cached = {}
    global_smoothed_means = []

    for group, seed_dirs in sorted(groups.items()):
        triple = extract_group_series(seed_dirs)
        if triple is None:
            continue
        xs, m, s, y_tag, x_tag = triple

        # IMPORTANT: compute min/max on the SAME signal used for plotting (smoothed mean)
        m_sm = _smooth(m, smooth_sigma)

        cached[group] = (xs, m, s, m_sm, y_tag, x_tag)
        global_smoothed_means.extend(m_sm.tolist())

    if not cached:
        print("[WARN] No chartable data found (no tfevents or no reward tag).")
        return

    # Global min/max based on smoothed mean (what goes into the plot)
    global_vmin, global_vmax = _safe_minmax(np.array(global_smoothed_means))

    print("[Charts] X auto (x_tag if exists else step).")
    print(f"[Charts] Normalization = MIN-MAX with scope={norm_scope}.")
    print(f"[Charts] Global min-max (on smoothed means): vmin={global_vmin:.6f}, vmax={global_vmax:.6f}")

    charts_root = os.path.join(results_dir, "charts")

    for group, (xs, m, s, m_sm, y_tag, x_tag) in cached.items():
        out_dir = os.path.join(charts_root, group)

        if norm_scope == "per_group":
            vmin, vmax = _safe_minmax(m_sm)
        else:
            vmin, vmax = global_vmin, global_vmax

        plot_raw(
            xs, m, s,
            title=f"{group} — raw (x={x_tag or 'step'}, y={y_tag})",
            save=os.path.join(out_dir, "raw.png"),
            smooth_sigma=smooth_sigma
        )

        plot_norm(
            xs, m, s,
            title=f"{group} — normalized (x={x_tag or 'step'}, y={y_tag})",
            save=os.path.join(out_dir, "normalized.png"),
            smooth_sigma=smooth_sigma,
            vmin=vmin,
            vmax=vmax
        )

        print(f"✔ charts: {group} | x={x_tag or 'step'} | y={y_tag} | norm vmin={vmin:.6f} vmax={vmax:.6f}")


# ----------------------------
# MAIN ENTRY (Notebook-safe)
# ----------------------------
def run_all(
    root_folder,
    results_folder_name=None,
    reward_col="episode_reward_mean",
    print_tree=True,
    do_charts=True,
    smooth_sigma=3.0,
    norm_scope="global",  # "global" or "per_group"
    show_seed_tables=False
):
    root_folder = os.path.abspath(root_folder)
    if not os.path.isdir(root_folder):
        raise ValueError(f"Folder not found: {root_folder}")

    base = os.path.basename(os.path.normpath(root_folder))
    if results_folder_name is None:
        results_folder_name = f"{base}_results"

    results_dir = os.path.join(os.path.dirname(root_folder), results_folder_name)
    os.makedirs(results_dir, exist_ok=True)

    if print_tree:
        print("📂 Folder tree:")
        print_tree_dir(root_folder)

    reward_grp, auc_grp, reward_seeds, auc_seeds = compute_reward_and_auc(root_folder, reward_col)

    reward_csv = os.path.join(results_dir, "reward_stats_grouped.csv")
    auc_csv = os.path.join(results_dir, "auc_grouped.csv")

    reward_grp.to_csv(reward_csv, index=False)
    auc_grp.to_csv(auc_csv, index=False)

    print("\n✅ Saved:")
    print(reward_csv)
    print(auc_csv)

    try:
        from IPython.display import display
        display(reward_grp)
        display(auc_grp)
        if show_seed_tables:
            display(reward_seeds.sort_values(["algo", "group", "seed"]))
            display(auc_seeds.sort_values(["algo", "group", "seed"]))
    except Exception:
        print("\n[Reward grouped]\n", reward_grp)
        print("\n[AUC grouped]\n", auc_grp)

    if do_charts:
        make_charts(
            root=root_folder,
            results_dir=results_dir,
            smooth_sigma=smooth_sigma,
            norm_scope=norm_scope
        )
        print("\n📊 Charts in:", os.path.join(results_dir, "charts"))

    print("\n[DONE] Results folder:", results_dir)
    return results_dir


In [2]:
# Mesma escala (0=min global, 1=max global) — bom pra comparar LSTM vs QRU:

run_all("./runs", "runs_results", print_tree=False, do_charts=True, norm_scope="global")


# Escala por grupo (0=min do grupo, 1=max do grupo) — bom pra ver “forma” de cada um:

# run_all("./runs", "runs_results", print_tree=False, do_charts=True, norm_scope="per_group")


✅ Saved:
/mnt/ssd1/rafael/graph_charts_article/runs_results/reward_stats_grouped.csv
/mnt/ssd1/rafael/graph_charts_article/runs_results/auc_grouped.csv


,algo,group,mean,std,count,mean_std
0,MAPPO,"MAPPO-QUANTUM-LSTM (0.5,0.7,0.5,0.6)",1025.098399,0.0,25,1025.098 ± 0.0
1,MAPPO,"MAPPO-QUANTUM-QRU (0.5,0.7,0.5,0.6)",999.658290,0.0,25,999.658 ± 0.0


,algo,group,mean,std,count,mean_std
0,MAPPO,"MAPPO-QUANTUM-LSTM (0.5,0.7,0.5,0.6)",407414.991375,0.0,25,407414.991 ± 0.0
1,MAPPO,"MAPPO-QUANTUM-QRU (0.5,0.7,0.5,0.6)",397301.437795,0.0,25,397301.438 ± 0.0


2026-02-01 19:01:02.278083: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-01 19:01:02.514335: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769972462.548164 2959148 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769972462.552747 2959148 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769972462.565013 2959148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`
[Charts] X auto (x_tag if exists else step).
[Charts] Normalization = MIN-MAX with scope=global.
[Charts] Global min-max (on smoothed means): vmin=141.628693, vmax=1125.094249
✔ charts: MAPPO-QUANTUM-LSTM (0.5,0.7,0.5,0.6) | x=timesteps_total | y=episode_reward_mean | norm vmin=141.628693 vmax=1125.094249
✔ charts: MAPPO-QUANTUM-QRU (0.5,0.7,0.5,0.6) | x=timesteps_total | y=episode_reward_mean | norm vmin=141.628693 vmax=1125.094249

📊 Charts in: /mnt/ssd1/rafael/graph_charts_article/runs_results/charts

[DONE] Results folder: /mnt/ssd1/rafael/graph_charts_article/runs_results


'/mnt/ssd1/rafael/graph_charts_article/runs_results'